## Simple Tool Calls
### Tools
**In LangChain Python, tools are created using the @tool decorator**
* Tools are nothing but functions/methods with proper defined input and output and description.
* These decriptions / docstrings added to functions helps LLM to understand what a Function/Tool does.
* Adding Pydantic schema as tool description allow LLM to understand about its strict schema.
* This function converted to Tool with @tool gives more context to LLM about the existing tools to it.

In [1]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

load_dotenv()

True

In [2]:
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable is not set.")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

model = ChatOpenAI(model="gpt-5-nano")

#### Define Pydantic schema for a Tool
* MovieShow decribes the structure of tool `scheck_showtime` defined below.
* This tool takes parameter movie_title. 

In [3]:
class MovieShows(BaseModel):
  movie_title : str = Field(description="MovieShows: Check available showtimes for a movie at the cinema")


### Defining a Tool
<img src="../../assets/tools_definition.png" width="800" height="300">

* Description in @tool decorator parameter overrides the docstring description.
* Langchain provides a way to define docstring and tool description separately, but is not provided in @tool, it will use docstring as desciption.
* We have created 2 tools `check_showtimes` and `reserve`.
* Here we also define `tools_name_func_map` that will help to make actual call to tools from our code.

In [4]:
@tool(args_schema=MovieShows)
def check_showtimes(movie_title: str) -> str:
  """Check available showtimes for a movie at the cinema.
  Args:
      movie_title: The exact title of the movie to check
  """
  fake_showtimes = {
      "interstellar": "7:00 PM and 10:15 PM",
      "dune part two": "9:30 PM only",
      "oppenheimer": "Sold out for tonight",
  }
  return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")

@tool('book_seats', description = 'Book Cinema for a customer, use whenever customer wants to book/reserve a seat.')
def reserve(movie:str, seats:int) -> str:
  """Reserve Seats"""
  return f"Reserved {seats} seat for {movie}"

print(json.dumps(check_showtimes.args, indent=2))


tools_name_func_map = {'book_seats': reserve, 'check_showtimes': check_showtimes}

{
  "movie_title": {
    "description": "MovieShows: Check available showtimes for a movie at the cinema",
    "title": "Movie Title",
    "type": "string"
  }
}


#### STEP 1: LLM Geneates Tool Call
* The LLM's role: Analyze the request and decide which tool to call
* Important: The LLM does NOT execute anything - it just generates a plan!

In [6]:
model_with_tools = model.bind_tools([check_showtimes, reserve])
query = "Is Interstellar showing tonight at 7pm at the Downtown cinema ?"
response = model_with_tools.invoke(query)

#### LLM response
* LLM responded to make a tool call with below information.
    1. Name
    2. Arguments
    3. Id 

In [7]:
print(response.content)
tool_call = response.tool_calls[0]
print("LLM decided to call:", tool_call["name"])
print("With arguments:", tool_call["args"])
print("Tool call ID:", tool_call["id"])


LLM decided to call: check_showtimes
With arguments: {'movie_title': 'Interstellar'}
Tool call ID: call_jVSPc8W7CBz9W1TZA9ytQQr2


#### STEP 2: Our Code Executes The Tool
* Our code actually execute the function and get real results
* This is where the real work happens - API calls, database queries, etc.
* Here we make call to tool `check_showtimes` using .invoke() method.
* Tool responded with return statement of check_showtimes. 

In [8]:
tool_result = tools_name_func_map[tool_call["name"]].invoke(tool_call["args"])
print("Tool executed successfully!")
print("Tool result:", tool_result)

Tool executed successfully!
Tool result: 7:00 PM and 10:15 PM


### STEP 3: Send Results Back To LLM.
* Create a messages list that contains our LLM conversation.
* Include 
    * HumanMessage with our query as content.
    * AIMessage with LLM response content and suggested tool to make call on.
    * ToolMessage with Tool Results and the tool call id.
* Give this information back to LLM to get the refined Natural Language response.

In [9]:
messages = [
    HumanMessage(content=query),
    AIMessage(content=str(response.content), tool_calls=response.tool_calls),
    ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])
]

#### Refined Final response
* Final reponse by LLM with information about available show time for a movie.
* **Also it responded with option to further reserve tickets, this is because LLM knows it has the capability (Tool) to reserve ticket**

In [10]:
final_response = model.invoke(messages)
print(final_response.content)

Yes. Interstellar is showing tonight at the Downtown cinema at 7:00 PM (also a 10:15 PM showing). Want me to reserve tickets or check seat availability for the 7:00 PM show?
